# Reproducing every number of the paper

Runs the notebooks of `data_load/` (only if the database does not exist), `methodology/` and `results/` in order, saving the outputs of each, and ends with one table. The notebooks hold the analysis; this one only calls them. It needs `raw_data/` (see the section Data of `README.md`).


In [1]:
import importlib
import json
import sys
import time
from pathlib import Path

import nbformat
from nbclient import NotebookClient

REPO_ROOT = Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
DATABASE = REPO_ROOT / "database" / "CSCAF_public.db"
RAW_DATA = REPO_ROOT / "raw_data"
NEEDED = ["countries.json", "annotation", "curricula", "job_advertisements", "referent_framework", "supplementary",
          "training_courses"]
DATA_LOAD = sorted((REPO_ROOT / "data_load").glob("*.ipynb"))
ANALYSIS = sorted((REPO_ROOT / "methodology").glob("*.ipynb")) + sorted((REPO_ROOT / "results").glob("*.ipynb"))

missing = [name for name in NEEDED if not (RAW_DATA / name).exists()]
if missing:
    raise SystemExit(f"raw_data is not in place (missing: {', '.join(missing)}). The data are available from the "
                     f"authors on reasonable request; see the section Data of README.md.")
if not DATABASE.exists():
    import torch
    if not torch.cuda.is_available():
        raise SystemExit("The database does not exist and building it needs a CUDA GPU; see README.md.")
print("raw_data :", "in place")
print("database :", "exists, data_load is skipped" if DATABASE.exists() else "will be built by data_load (about one hour)")


raw_data : in place
database : exists, data_load is skipped


In [2]:
def run(path):
    """Run one notebook from its own folder and save its outputs. Returns (seconds, error)."""
    notebook, started, error = nbformat.read(path, as_version=4), time.time(), None
    try:
        NotebookClient(notebook, kernel_name="python3", timeout=None,
                       resources={"metadata": {"path": str(path.parent)}}).execute()
    except Exception as exception:
        error = type(exception).__name__ + ": " + str(exception)[-400:]
    nbformat.write(notebook, path)
    return time.time() - started, error


report = []
for path in (DATA_LOAD if not DATABASE.exists() else []) + ANALYSIS:
    seconds, error = run(path)
    report.append((path.relative_to(REPO_ROOT).as_posix(), seconds, error))
    print(f"{report[-1][0]}: {seconds:.0f} s" + (" ERROR" if error else ""), flush=True)
    if error and path in DATA_LOAD:
        raise SystemExit(f"{path.name} failed, the database is not complete: {error}")


methodology/01_reference_framework.ipynb: 5 s


methodology/02_data.ipynb: 2 s


methodology/03_text_processing.ipynb: 16 s


methodology/04_threshold_calibration.ipynb: 16 s


results/05_role_level_results.ipynb: 84 s


results/06_curriculum_update.ipynb: 5 s


results/07_employer_demand.ipynb: 57 s


In [3]:
def compare(path):
    """Every number of the record of one notebook against the paper. Returns (rows, rows that differ)."""
    module = importlib.import_module(f"paper_checks.check_{path.stem}")
    record = json.loads((path.parent / "output" / f"{path.stem}.json").read_text(encoding="utf-8"))
    rows = module.rows(record)
    return rows, [row for row in rows if row[1] != row[2]]


compared = {path: compare(path) for path in ANALYSIS}


In [4]:
errors = {name: error for name, _, error in report}
seconds_of = {name: seconds for name, seconds, _ in report}
print(f"{'notebook':48s} {'seconds':>7s}  {'as in the paper':>15s}  different  error")
for name, seconds, error in report:
    if name.startswith("data_load"):
        print(f"{name:48s} {seconds:7.0f}  {'-':>15s}  {'-':>9s}  {error or '-'}")
for path, (rows, different) in compared.items():
    name = path.relative_to(REPO_ROOT).as_posix()
    same = f"{len(rows) - len(different)} of {len(rows)}"
    print(f"{name:48s} {seconds_of.get(name, 0):7.0f}  {same:>15s}  {len(different):9d}  {errors.get(name) or '-'}")
for path, (_, different) in compared.items():
    for name, printed, computed in different:
        print(f"\n{path.stem}: {name}\n  paper: {printed}\n  now  : {computed}")
figures = sorted(p.relative_to(REPO_ROOT).as_posix() for folder in ("methodology", "results")
                 for p in (REPO_ROOT / folder / "figures").glob("*.png"))
print(f"\nfigures: {len(figures)}")
print("\n".join("  " + figure for figure in figures))
print(f"\nnumbers compared with the paper: {sum(len(rows) for rows, _ in compared.values())}, "
      f"different: {sum(len(different) for _, different in compared.values())}; "
      f"notebooks with an error: {sum(1 for error in errors.values() if error)}")


notebook                                         seconds  as in the paper  different  error
methodology/01_reference_framework.ipynb               5         21 of 21          0  -
methodology/02_data.ipynb                              2         47 of 47          0  -
methodology/03_text_processing.ipynb                  16           4 of 4          0  -
methodology/04_threshold_calibration.ipynb            16         39 of 39          0  -
results/05_role_level_results.ipynb                   84         52 of 52          0  -
results/06_curriculum_update.ipynb                     5         48 of 48          0  -
results/07_employer_demand.ipynb                      57       100 of 100          0  -

figures: 7
  methodology/figures/figure_2_threshold_calibration.png
  results/figures/figure_3_role_coverage.png
  results/figures/figure_4_role_demand.png
  results/figures/figure_5_skills_against_knowledge.png
  results/figures/figure_6_programme_position.png
  results/figures/figure_7_co